In [ ]:
from IPython.display import HTML, display
display(HTML('''
<style>
  body, .jp-Notebook { background: #eceff1 !important; }
  .jp-Cell-outputWrapper, .output_area { background: transparent !important; }
  .jp-OutputArea-child { background: transparent !important; }
  div.output_html { background: transparent !important; }
</style>
'''))
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import ugbio_mrd.mrd_utils as mrd
from ugbio_mrd.mrd_detection import run_detection_analysis, format_scientific, plot_null_distribution

In [ ]:
from ugbio_core.plotting_utils import set_pyplot_defaults

set_pyplot_defaults(
    title_size=12,
    small_size=10,
    medium_size=10,
    bigger_size=11,
)
%matplotlib inline

In [ ]:
# input parameters
features_file_parquet = None
signatures_file_parquet = None
signature_filter_query_default = "(norm_coverage <= 2.5) and (norm_coverage >= 0.6)"
signature_filter_query = signature_filter_query_default
read_filter_query_default = "filt>0 and snvq>60 and mapq>=60"
read_filter_query = read_filter_query_default
featuremap_df_file = None
srsnv_metadata_json = None
output_dir = None
basename = None
show_raw_tables = False


In [ ]:
if features_file_parquet is None:
    raise ValueError("Required input features_file_parquet not provided")
if signatures_file_parquet is None:
    raise ValueError("Required input signatures_file_parquet not provided")
if featuremap_df_file is None:
    raise ValueError("Required input featuremap_df_file not provided")

In [ ]:
# read and filter df_features
df_features, df_features_filt, filtering_ratio = mrd.read_and_filter_features_parquet(
    features_file_parquet, read_filter_query
)

In [ ]:
# read and filter df_signatures
df_signatures, df_signatures_filt = mrd.read_and_filter_signatures_parquet(
    signatures_file_parquet, signature_filter_query, filtering_ratio
)

In [ ]:
denom_ratio, filt_ratio, read_filter_non_filt = mrd.calc_tumor_fraction_denominator_ratio(
    featuremap_df_file, srsnv_metadata_json, read_filter_query
)

In [ ]:
df_tf_filt, df_supporting_reads_per_locus_filt = mrd.get_tf_from_filtered_data(
    df_features_filt,
    df_signatures_filt,
    plot_results=False,
    title="Filtered reads and signatures",
    denom_ratio=denom_ratio,
)

In [ ]:
# Compute secondary analyses (data only, displayed in sections 2 & 3)
df_tf_unfilt, df_supporting_reads_per_locus_unfilt = mrd.get_tf_from_filtered_data(
    df_features_filt, df_signatures, plot_results=False,
    title='Filtered reads, unfiltered signatures', denom_ratio=denom_ratio,
)
detection_unfilt = run_detection_analysis(
    df_tf=df_tf_unfilt, df_signatures_filt=df_signatures, denom_ratio=denom_ratio
)
df_tf_unfilt2, df_supporting_reads_per_locus_unfilt2 = mrd.get_tf_from_filtered_data(
    df_features, df_signatures_filt, plot_results=False,
    title='Unfiltered reads, filtered signatures', denom_ratio=1,
)
detection_unfilt2 = run_detection_analysis(
    df_tf=df_tf_unfilt2, df_signatures_filt=df_signatures_filt, denom_ratio=1
)


In [ ]:
# Run statistical detection analysis
detection = run_detection_analysis(
    df_tf=df_tf_filt,
    df_signatures_filt=df_signatures_filt,
    denom_ratio=denom_ratio,
)

In [ ]:
# Display detection result with mockup-style blue header and detection banner
from IPython.display import HTML, display
import datetime

if detection.detected is True:
    banner_color = '#c0392b'; banner_bg = '#fdedec'; banner_border = '#e74c3c'
elif detection.detected is False:
    banner_color = '#1e8449'; banner_bg = '#eafaf1'; banner_border = '#2ecc71'
else:
    banner_color = '#b7950b'; banner_bg = '#fef9e7'; banner_border = '#f1c40f'

lod_str = format_scientific(detection.personal_lod) if detection.personal_lod else 'N/A'
vaf_str = format_scientific(detection.matched_ctdna_vaf) if detection.matched_ctdna_vaf > 0 else '0'
report_date = datetime.date.today().isoformat()
ep = detection.p_value
emp_p_str = f'{ep:.3f}' if ep >= 0.001 else f'{ep:.2e}'
fp = detection.fitted_p_value
fit_p_str = (f'{fp:.3f}' if fp >= 0.001 else f'{fp:.2e}') if fp is not None else None
fit_p_html = (
    f'<div>'
    f'<div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.8px;">p-value (Poisson fit)</div>'
    f'<div style="font-size:18px;font-weight:700;color:#2c3e50;">{fit_p_str}</div>'
    f'<div style="font-size:11px;color:#7f8c8d;">Poisson-fitted null</div>'
    f'</div>'
) if fit_p_str is not None else ''

html = (
    '<div style="max-width:960px;margin:0 auto;background:#fff;border-radius:8px;'
    'box-shadow:0 2px 12px rgba(0,0,0,.12);overflow:hidden;font-family:\'Segoe UI\',system-ui,sans-serif;">'
    '<div style="background:#1a3a5c;color:#fff;padding:20px 28px;'
    'display:flex;align-items:center;justify-content:space-between;">'
    '<div>'
    '<div style="font-size:20px;font-weight:700;color:#1abc9c;letter-spacing:1px;margin-bottom:4px;">ULTIMA GENOMICS</div>'
    '<div style="font-size:17px;font-weight:600;letter-spacing:.3px;">MRD Analysis Report</div>'
    '<div style="font-size:11px;color:rgba(255,255,255,.65);margin-top:2px;">'
    'Tumor-Informed Minimal Residual Disease Detection \u00b7 Whole Genome Sequencing</div>'
    '</div>'
    f'<div style="text-align:right;font-size:11px;color:rgba(255,255,255,.7);line-height:1.9;">'
    f'<strong style="color:#fff;">Sample</strong> {basename or "N/A"}<br>'
    f'<strong style="color:#fff;">Report Date</strong> {report_date}'
    '</div></div>'
    '<div style="padding:20px 28px;">'
    '<div style="font-size:10px;font-weight:700;letter-spacing:1.2px;text-transform:uppercase;'
    'color:#7f8c8d;margin-bottom:10px;padding-bottom:6px;border-bottom:1px solid #dde1e7;">'
    'Detection Result</div>'
    f'<div style="background:{banner_bg};border:1.5px solid {banner_border};'
    f'border-left:5px solid {banner_color};border-radius:6px;'
    'padding:16px 22px;display:flex;align-items:center;gap:24px;flex-wrap:wrap;">'
    f'<div style="background:{banner_color};color:white;font-size:13px;font-weight:700;'
    'padding:6px 14px;border-radius:4px;text-transform:uppercase;letter-spacing:.8px;white-space:nowrap;">'
    f'{detection.call}</div>'
    '<div style="display:flex;gap:24px;flex-wrap:wrap;">'
    '<div><div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.8px;">Tumor Fraction</div>'
    f'<div style="font-size:18px;font-weight:700;color:#2c3e50;">{vaf_str}</div></div>'
    '<div><div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.8px;">p-value (empirical)</div>'
    f'<div style="font-size:18px;font-weight:700;color:#2c3e50;">{emp_p_str}</div>'
    f'<div style="font-size:11px;color:#7f8c8d;">{detection.n_synthetic_controls} synthetic controls</div></div>'
    f'{fit_p_html}'
    '<div><div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.8px;">Personal LOD (95%)</div>'
    f'<div style="font-size:18px;font-weight:700;color:#2c3e50;">{lod_str}</div>'
    '<div style="font-size:11px;color:#7f8c8d;">95% detection power</div></div>'
    '<div><div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.8px;">ctDNA Reads Supporting Signature</div>'
    f'<div style="font-size:18px;font-weight:700;color:#2c3e50;">{detection.matched_supporting_reads}</div>'
    f'<div style="font-size:11px;color:#7f8c8d;">noise median: {detection.null_median_reads:.1f}</div></div>'
    + (f'<div style="margin-top:10px;padding:8px 12px;background:#fef9e7;border:1px solid #f1c40f;'
       f'border-radius:4px;font-size:11px;color:#7f8c8d;">'
       f'⚠️ Only {detection.n_synthetic_controls} synthetic controls — '
       f'p-value reliability is reduced. Consider adding more synthetic controls for better MRD detection.</div>'
       if detection.n_synthetic_controls < 20 else '')
    + '</div></div></div></div>'
)
intro_div = ('<div style="padding:6px 28px 14px;font-size:12px;color:#5d6d7e;font-family:\'Segoe UI\',system-ui,sans-serif;border-top:1px solid rgba(255,255,255,0.15);">This report presents results of tumor-informed MRD analysis for one cfDNA sample against a matched tumor signature and a set of synthetic database controls. Three analyses are shown: (1)\u00a0filtered reads & filtered signature (primary result), (2)\u00a0impact of signature QC filters, (3)\u00a0impact of read quality filters.</div>')
html = html.replace('</div>\n</div>', intro_div + '</div>\n</div>', 1)
display(HTML(html))


In [ ]:
# Assay metrics summary
metrics_html = f"""
<div style="display:grid; grid-template-columns:repeat(4, 1fr); gap:12px;
     margin:10px 0; font-family:sans-serif;">
  <div style="background:#f4f6f8; border:1px solid #dde1e7; border-radius:6px; padding:14px 16px;">
    <div style="font-size:10px; color:#7f8c8d; text-transform:uppercase;">Signature Size</div>
    <div style="font-size:20px; font-weight:700; color:#1a3a5c;">{detection.signature_size:,}</div>
    <div style="font-size:11px; color:#7f8c8d;">loci after QC filters</div>
  </div>
  <div style="background:#f4f6f8; border:1px solid #dde1e7; border-radius:6px; padding:14px 16px;">
    <div style="font-size:10px; color:#7f8c8d; text-transform:uppercase;">Mean Coverage</div>
    <div style="font-size:20px; font-weight:700; color:#1a3a5c;">{detection.mean_coverage:.0f}&times;</div>
    <div style="font-size:11px; color:#7f8c8d;">at signature loci</div>
  </div>
  <div style="background:#f4f6f8; border:1px solid #dde1e7; border-radius:6px; padding:14px 16px;">
    <div style="font-size:10px; color:#7f8c8d; text-transform:uppercase;">Noise Floor</div>
    <div style="font-size:20px; font-weight:700; color:#1a3a5c;">{format_scientific(detection.null_median_reads / detection.corrected_coverage) if detection.corrected_coverage > 0 else 'N/A'}</div>
    <div style="font-size:11px; color:#7f8c8d;">median of {detection.n_synthetic_controls} synthetic controls</div>
  </div>
  <div style="background:#f4f6f8; border:1px solid #dde1e7; border-radius:6px; padding:14px 16px;">
    <div style="font-size:10px; color:#7f8c8d; text-transform:uppercase;">Pass Rate</div>
    <div style="font-size:20px; font-weight:700; color:#1a3a5c;">{denom_ratio:.2f}</div>
    <div style="font-size:11px; color:#7f8c8d;">fraction of reads passing quality filter</div>
  </div>
</div>
"""
display(HTML(metrics_html))

In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:0 0 18px;padding:10px 28px 8px;background:#f4f6f8;border-left:4px solid #1abc9c;
     font-family:'Segoe UI',system-ui,sans-serif;font-size:12px;color:#5d6d7e;">
  <strong style="color:#1a3a5c;">MRD QC Report</strong> &mdash; Quality control diagnostics: 
impact of signature and read filters on ctDNA detection.
</div>
'''))


In [ ]:
# Mutation-type profile function (6 substitution classes, horizontal)
_SBS_TYPES  = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
_SBS_COLORS = ["#85c7e8", "#6b6b6b", "#e88585", "#d6d6d6", "#b5d98a", "#f4d0cd"]
_COMP       = str.maketrans("ACGT", "TGCA")

def plot_sbs_profile(df_sig, title="", ax=None, query=None):
    """Plot horizontal 6-class mutation-type profile from signature dataframe."""
    df = df_sig if query is None else df_sig.query(query)
    _all_muts = ["C->A", "C->G", "C->T", "T->A", "T->C", "T->G"]
    counts = df["mutation_type"].value_counts(normalize=True).reindex(_all_muts, fill_value=0)
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 2.4))
    bars = ax.barh(range(6), counts.values[::-1], color=list(reversed(_SBS_COLORS)),
                   height=0.65, linewidth=0)
    ax.set_yticks(range(6))
    ax.set_yticklabels([m.replace("->", ">") for m in reversed(_all_muts)],
                       fontsize=9, fontweight="bold")
    for j, (bar, val) in enumerate(zip(bars, counts.values[::-1])):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f"{val:.1%}", va="center", fontsize=8, color="#555")
    ax.set_xlim(0, max(counts.values) * 1.25 + 0.02)
    ax.set_xlabel("Fraction", fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.text(0.99, 0.02, f"n = {len(df):,}", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=8, color="#7f8c8d")
    ax.spines[["top", "right"]].set_visible(False)
    return ax


In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:28px 0 8px;padding-bottom:7px;border-bottom:2px solid #dde1e7;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:14px;font-weight:700;letter-spacing:1.2px;text-transform:uppercase;color:#7f8c8d;">Control Signature Profiles</span>
  <p style="font-size:12px;color:#5d6d7e;margin:5px 0 0;">All cohort controls and one representative synthetic control.</p>
</div>
'''))


In [ ]:
# All cohort controls + 1 synthetic control
_cohort_sigs = sorted(df_signatures.query("signature_type == 'control'")['signature'].unique())
_syn_sigs    = sorted(df_signatures.query("signature_type == 'db_control'")['signature'].unique())[:1]
_ctrl_sigs   = _cohort_sigs + _syn_sigs
for _sig in _ctrl_sigs:
    _df  = df_signatures.query(f"signature == '{_sig}'")
    _typ = _df['signature_type'].iloc[0]
    _label = f'{_sig}  ({_typ})'
    from IPython.display import HTML, display
    display(HTML(f'<div style="font-size:11px;font-weight:600;color:#5d6d7e;font-family:sans-serif;margin:10px 0 2px;padding:3px 0;border-bottom:1px solid #eaeef2;">{_label}</div>'))
    fig, axs = plt.subplots(1, 2, figsize=(12, 2.4))
    plot_sbs_profile(_df, title='Unfiltered', ax=axs[0])
    plot_sbs_profile(_df, title='After QC filters', ax=axs[1], query=signature_filter_query)
    plt.tight_layout()
    plt.show()
    try:
        mrd.plot_signature_allele_fractions(_df, signature_filter_query)
    except Exception:
        pass


In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:28px 0 8px;padding-bottom:7px;border-bottom:2px solid #dde1e7;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:14px;font-weight:700;letter-spacing:1.2px;text-transform:uppercase;color:#7f8c8d;">cfDNA Fragment Length Distributions</span>
  
</div>
'''))


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(20, 10), sharex=True)
fig.subplots_adjust(hspace=0.4, wspace=0.3)

max_value = 0
for ax, title, x in zip(
    axs.flatten(),
    [
        "Matched reads (from tumor)\nunfiltered",
        "Matched reads (from tumor)\nfiltered",
        "Unmatched reads (not tumor)\nunfiltered",
        "Unmatched reads (not tumor)\nfiltered",
    ],
    [
        df_features.query("signature_type!='matched'")["rl"],
        df_features.query(f"signature_type!='matched' and {read_filter_query}")["rl"],
        df_features.query("signature_type!='matched'")["rl"],
        df_features.query(f"signature_type!='matched' and {read_filter_query}")["rl"],
    ],
    strict=False,
):
    plt.sca(ax)
    plt.title(title, y=1.05, fontsize=28)
    max_value = max(max_value, x.max())
    x.plot.hist(bins=np.arange(0.5, max(250, max_value)))
for ax in axs[-1, :]:
    ax.set_xlabel("Read length", fontsize=32)

Distribution of read lengths for cfDNA reads, both matched and unmatched. Not all of the reads are sequenced through, so the longer reads might be limited by read rather than insert length. Differences in the distributions between matched and unmatched reads could be used for more refined filtering of reads.

In [ ]:
## Write secondary tf tables to hdf file
output_h5_file = os.path.join(output_dir, basename + ".ctdna_vaf.h5")
for key, val in {
    "df_ctdna_vaf_unfilt_signature_filt_featuremap": df_tf_unfilt,
    "df_ctdna_vaf_filt_signature_unfilt_featuremap": df_tf_unfilt2,
    "df_supporting_reads_per_locus_unfilt_signature_filt_featuremap": df_supporting_reads_per_locus_unfilt,
    "df_supporting_reads_per_locus_filt_signature_unfilt_featuremap": df_supporting_reads_per_locus_unfilt2,
}.items():
    val.to_hdf(output_h5_file, key=key, mode="a")

In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:28px 0 8px;padding-bottom:7px;border-bottom:2px solid #dde1e7;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:14px;font-weight:700;letter-spacing:1.2px;text-transform:uppercase;color:#7f8c8d;">QC Analysis — Unfiltered Signature</span>
  <p style="font-size:12px;color:#5d6d7e;margin:5px 0 0;">Filtered reads against all signature loci before QC filtering. Highlights the impact of coverage and blacklist filters on the signature.</p>
</div>
'''))


In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:18px 0 4px;padding-bottom:4px;border-bottom:1px solid #eaeef2;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:12px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:#aab4be;">Signal vs. Noise</span>
</div>
'''))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_null_distribution(detection_unfilt, df_tf_unfilt, ax=ax)
plt.tight_layout(); plt.show()


In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:18px 0 4px;padding-bottom:4px;border-bottom:1px solid #eaeef2;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:12px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:#aab4be;">Mutational Profile</span>
</div>
'''))


In [ ]:
for _sig in df_signatures.query("signature_type == 'matched'")['signature'].unique():
    _df = df_signatures.query(f"signature == '{_sig}'")
    fig, ax = plt.subplots(figsize=(6, 2.8))
    plot_sbs_profile(_df, title='All loci (unfiltered)', ax=ax)
    plt.tight_layout()
    plt.show()


In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:18px 0 4px;padding-bottom:4px;border-bottom:1px solid #eaeef2;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:12px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:#aab4be;">Signature VAF</span>
</div>
'''))


In [ ]:
for _sig in df_signatures.query("signature_type == 'matched'")['signature'].unique():
    _df = df_signatures.query(f"signature == '{_sig}'")
    mrd.plot_signature_allele_fractions(_df, signature_filter_query, panel='unfiltered')


In [ ]:
mrd.plot_vaf_matched_unmatched(df_supporting_reads_per_locus_unfilt, df_signatures)
if show_raw_tables:
    display(df_tf_unfilt)


In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:28px 0 8px;padding-bottom:7px;border-bottom:2px solid #dde1e7;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:14px;font-weight:700;letter-spacing:1.2px;text-transform:uppercase;color:#7f8c8d;">QC Analysis — Unfiltered Reads</span>
  <p style="font-size:12px;color:#5d6d7e;margin:5px 0 0;">All reads against QC-filtered signature loci. Highlights the impact of read quality filters on ctDNA detection.</p>
</div>
'''))


In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:18px 0 4px;padding-bottom:4px;border-bottom:1px solid #eaeef2;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:12px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:#aab4be;">Signal vs. Noise</span>
</div>
'''))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_null_distribution(detection_unfilt2, df_tf_unfilt2, ax=ax)
plt.tight_layout(); plt.show()


In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:18px 0 4px;padding-bottom:4px;border-bottom:1px solid #eaeef2;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:12px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:#aab4be;">Mutational Profile</span>
</div>
'''))


In [ ]:
for _sig in df_signatures_filt.query("signature_type == 'matched'")['signature'].unique():
    _df = df_signatures_filt.query(f"signature == '{_sig}'")
    fig, ax = plt.subplots(figsize=(6, 2.8))
    plot_sbs_profile(_df, title='QC-filtered signature', ax=ax)
    plt.tight_layout()
    plt.show()


In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="margin:18px 0 4px;padding-bottom:4px;border-bottom:1px solid #eaeef2;font-family:'Segoe UI',system-ui,sans-serif;">
  <span style="font-size:12px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:#aab4be;">Signature VAF</span>
</div>
'''))


In [ ]:
for _sig in df_signatures_filt.query("signature_type == 'matched'")['signature'].unique():
    _df = df_signatures_filt.query(f"signature == '{_sig}'")
    mrd.plot_signature_allele_fractions(_df, signature_filter_query, panel='filtered')


In [ ]:
mrd.plot_vaf_matched_unmatched(df_supporting_reads_per_locus_unfilt2, df_signatures_filt)
if show_raw_tables:
    display(df_tf_unfilt2)
